In [ ]:
import os

# Change this variable if the root folder name has been changed
root_dir = "nvae-shape-encoding"
current_dir = os.getcwd()

if not current_dir.endswith(root_dir):
    %cd ../..

assert os.getcwd().endswith(root_dir)

In [ ]:
import lightning as L
import torch

from arch.nvae.nvae import NVAE
from const import SEED
from data_modules.acdc import ACDCMaskDataModule
from utils.utils import setup_device

model_path = "logs/nvae_acdc/big-latent-space/pc-4-ws-6420-b-1/checkpoints/epoch=98-step=21186.ckpt"

# Setup device
device = setup_device()
print(f"Device: {device}")

# Seed
L.seed_everything(SEED)

# Load data
data_module = ACDCMaskDataModule(batch_size=20)

# Reseed after preprocessing data
L.seed_everything(SEED)

# Load model
model = NVAE.load_from_checkpoint(model_path)

In [ ]:
from utils.utils import show_samples

with torch.no_grad():
    model.eval()
    model.to(device)
    
    x_fake = model.decoder.generate(num_samples=40, device=device, temp=0.9)
    feats_fake = model.conditional_coder(x_fake)

    # Discretise probabilistic map then view generations
    generations = torch.argmax(feats_fake, dim=1).unsqueeze(1)
    show_samples(generations, rgb=False, ncol=10, figsize=(10, 4))